In [17]:
import pandas as pd
import numpy as np
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

rows = """1900,37.8
1905,39.8
1910,33.9
1914,42
1916,103.1
1919,139.2
1921,174.5
1923,183.5
1926,174.2
1930,178.3
1933,189.6
1936,155.7
1939,136.6
1942,176.6
1944,233.3
1946,251.7
1948,210.6
1950,185.6
1953,152.6
1956,123.8
1960,102.4
1965,80.8
1970,54.7
1975,49.3
1980,40.3
1985,37
1990,21.6
1993,31
1996,36.4
2000,28.2
2004,33.4
2008,50.2
2010,70.6
2013,78.8
2016,82.7
2019,84.5
2020,95.4
2022,93.2
2024,93.2
2025,93.8"""
df = pd.DataFrame([r.split(",") for r in rows.strip().split("\n")], columns=["year", "value"])
df["year"] = df["year"].astype(int)
df["value"] = df["value"].astype(float)
df["date"] = pd.to_datetime(df["year"], format="%Y")
df["series"] = "debt"

# OBR July 2026 FRS projections (replace with exact 5.3 annual values if you have them)
yrs = list(range(2026, 2076))
base_v  = np.interp(yrs, [2026, 2035, 2045, 2055, 2065, 2075], [95, 98, 117, 154, 215, 300])
worst_v = np.interp(yrs, [2026, 2035, 2045, 2055, 2065, 2075], [95, 131, 207, 334, 563, 1019])
base  = pd.DataFrame({"year": yrs, "value": base_v})
worst = pd.DataFrame({"year": yrs, "value": worst_v})
for d in (base, worst):
    d["date"] = pd.to_datetime(d["year"], format="%Y")

# slider — at 2025 only history shows and the axis fits it (identical to the original chart)
slider = alt.binding_range(min=2025, max=2075, step=1, name="Show to year: ")
endYear = alt.param(name="endYear", value=2025, bind=slider)

events = pd.DataFrame([
    (1914, "WWI",              "left",  8),
    (1939, "WWII",             "left",  8),
    (2008, "GFC", "right", 8),
    (2020, "COVID-19",         "left", 8),
], columns=["year", "label", "lab_align", "ypx"])
events["date"] = pd.to_datetime(events["year"], format="%Y")

ev_rule = alt.Chart(events).mark_rule(
    color="#E6224B", strokeDash=[3, 3], strokeWidth=1.2).encode(x="date:T")

left_txt = alt.Chart(events[events["lab_align"] == "left"]).mark_text(
    align="left", dx=4, fontSize=10, fontWeight="bold", color="#E6224B").encode(
    x="date:T", y=alt.Y("ypx:Q", scale=None), text="label:N")
right_txt = alt.Chart(events[events["lab_align"] == "right"]).mark_text(
    align="right", dx=-4, fontSize=10, fontWeight="bold", color="#E6224B").encode(
    x="date:T", y=alt.Y("ypx:Q", scale=None), text="label:N")

# history line — no fixed y-max so the axis rescales as projections are revealed
line = alt.Chart(df).mark_line(strokeWidth=1.4).encode(
    x=alt.X("date:T", axis=alt.Axis(format="%Y", tickCount=7), title=None),
    y=alt.Y("value:Q", scale=alt.Scale(domainMin=0), title="Net debt (% of GDP)"),
    color=alt.Color("series:N", legend=None))

# projections — revealed up to the slider year; eco colours, distinct from the red labels
base_line = alt.Chart(base).transform_filter(alt.datum.year <= endYear).mark_line(
    strokeWidth=1.8, strokeDash=[4, 3], color="#36B7B4").encode(
    x="date:T", y="value:Q")
worst_line = alt.Chart(worst).transform_filter(alt.datum.year <= endYear).mark_line(
    strokeWidth=1.8, strokeDash=[4, 3], color="#5C267B").encode(
    x="date:T", y="value:Q")

caption = alt.Title(
    text="Source: ONS / OBR (July 2026 Fiscal Risks and Sustainability report)",
    subtitle=[
        "Public sector net debt, % of GDP. History to 2025; drag to reveal the OBR projection.",
        "As the projection climbs, even the world wars shrink to blips.",
    ],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (ev_rule + left_txt + right_txt + line + base_line + worst_line)
    .add_params(endYear)
    .properties(width=700, height=270, title=caption)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor="#676A86", titleColor="#676A86"))

chart

alt.LayerChart(...)